# Countdown × GRPO — Qwen3.5-0.8B-Base, no SFT

Runner for `countdown/GUIDE.md` (the "48+63=111" TinyZero recipe on a 2026 0.8B base model).
Self-contained: the four `.py` files are written to disk by cells below — nothing to clone.

**Runtime → Change runtime type → A100** (40 GB). L4 works with the smaller batch noted in the train cell.

Order: install → self-tests → **baseline eval (go/no-go, ~10 min)** → 5-step timing run → 400-step train → curves → after-eval.

In [ ]:
!nvidia-smi
import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

In [ ]:
%%capture
!pip install -U "trl>=1.11" "transformers>=5.2" datasets peft accelerate wandb matplotlib
!pip install flash-linear-attention          # Qwen3.5 Gated-DeltaNet kernels (Triton). Without it: silent slow path.
# Optional & minor (often fails to build on Colab; skip if so): !pip install causal-conv1d --no-build-isolation
# Optional fast generation:     !pip install "vllm==0.27.1"   -> then add --use_vllm to the train command

In [ ]:
# Must print "fla OK" — transformers 5.16 gives NO warning when the fast kernels are missing.
!python -c "from fla.ops.gated_delta_rule import chunk_gated_delta_rule; print('fla OK')"
import trl, transformers, datasets; print("trl", trl.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)

## Code (embedded copies of `countdown/*.py`; regenerate with `build_colab_notebook.py` after edits)

In [ ]:
!mkdir -p /content/grpo_puzzles/countdown
%cd /content/grpo_puzzles/countdown

In [ ]:
%%writefile /content/grpo_puzzles/countdown/data.py
"""Dataset + prompt construction for Countdown GRPO.

Source: Jiayi-Pan/Countdown-Tasks-3to4  (490,364 rows; columns `target` int, `nums` list[int])
  * 240,632 rows have 3 numbers, 249,732 have 4 numbers
  * targets are 10..100, operands 1..99

We use the TinyZero *base-model* template verbatim (no chat template, no system role).
The prompt ends with "<think>" so the model's completion starts inside the think block.
"""

from __future__ import annotations

from datasets import Dataset, load_dataset

DATASET_ID = "Jiayi-Pan/Countdown-Tasks-3to4"
N_EVAL = 500          # held-out puzzles, never seen in training (taken from the shuffled head)
SEED = 42

# NOTE: identical to TinyZero examples/data_preprocess/countdown.py, template_type='base'
PROMPT_TEMPLATE = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\n"
    "User: Using the numbers {nums}, create an equation that equals {target}. "
    "You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. "
    "Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, "
    "for example <answer> (1 + 2) / 3 </answer>.\n"
    "Assistant: Let me solve this step by step.\n"
    "<think>"
)


def build_prompt(nums: list[int], target: int) -> str:
    return PROMPT_TEMPLATE.format(nums=list(nums), target=target)


def _to_prompt_row(ex):
    return {"prompt": build_prompt(ex["nums"], ex["target"]), "target": ex["target"], "nums": ex["nums"]}


def load_countdown(n_train: int = 20_000, curriculum: str = "none", seed: int = SEED):
    """Return (train_ds, eval_ds) with columns prompt / target / nums.

    curriculum:
      "none"   -> mix of 3- and 4-number puzzles (the raw distribution)
      "3only"  -> only 3-number puzzles (much easier; use this if the base model shows ~0% solve rate)
      "4only"  -> only 4-number puzzles (harder second stage)
    The eval split is always the raw mix so numbers are comparable across runs.
    """
    ds = load_dataset(DATASET_ID, split="train").shuffle(seed=seed)
    eval_ds = ds.select(range(N_EVAL))
    pool = ds.select(range(N_EVAL, len(ds)))
    if curriculum == "3only":
        pool = pool.filter(lambda ex: len(ex["nums"]) == 3)
    elif curriculum == "4only":
        pool = pool.filter(lambda ex: len(ex["nums"]) == 4)
    elif curriculum != "none":
        raise ValueError(f"unknown curriculum {curriculum!r}")
    train_ds = pool.select(range(min(n_train, len(pool))))
    return train_ds.map(_to_prompt_row), eval_ds.map(_to_prompt_row)


if __name__ == "__main__":
    tr, ev = load_countdown(n_train=100, curriculum="3only")
    print(tr)
    print(ev)
    print("---- example prompt ----")
    print(tr[0]["prompt"])
    print("---- target / nums ----", tr[0]["target"], tr[0]["nums"])


In [ ]:
%%writefile /content/grpo_puzzles/countdown/rewards.py
"""Rule-based rewards for the Countdown task (TRL GRPOTrainer signature).

TRL calls each reward function as  fn(prompts, completions, **dataset_columns)
where every non-`prompt` column of the dataset arrives as a keyword list
(here: `target` and `nums`).  Each function returns one float per completion.

Two separate rewards so that TRL logs them separately
(`rewards/format_reward/mean`, `rewards/answer_reward/mean`):

  format_reward : 1.0  -> "...</think>\n<answer>EQ</answer>" and NOTHING after
                  0.5  -> tags present in the right order but junk after </answer>
                  0.0  -> otherwise
  answer_reward : 1.0  -> equation uses every given number exactly once,
                          only + - * / ( ), and evaluates to target
                  0.0  -> otherwise

Total per completion is therefore in [0, 2].  The metric you care about is
`rewards/answer_reward/mean` == fraction of completions that solve the puzzle.

Reference implementations this is distilled from:
  * TinyZero  verl/utils/reward_score/countdown.py  (score 1.0 / format 0.1 / 0)
  * Phil Schmid Mini-R1 notebook                    (format 0/1 + equation 0/1)
"""

from __future__ import annotations

import re

# The prompt ends with "<think>", so the completion starts INSIDE the think block.
_FORMAT_RE = re.compile(
    r"^(?:(?!</think>).)*?</think>\s*<answer>(?:(?!</answer>).)*?</answer>(?P<tail>.*)$",
    re.DOTALL,
)
_ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
_ALLOWED_EQ = re.compile(r"^[\d+\-*/().\s]+$")


def _completion_text(c) -> str:
    """Accept both TRL formats: plain string (standard) or [{'role','content'}] (conversational)."""
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return c[0].get("content", "")
    return str(c)


def extract_equation(text: str) -> str | None:
    """First <answer>...</answer> block, stripped. None if absent."""
    m = _ANSWER_RE.search(text)
    return m.group(1).strip() if m else None


def equation_is_correct(equation: str | None, nums: list[int], target: int) -> bool:
    if not equation:
        return False
    if not _ALLOWED_EQ.match(equation):          # letters, names, etc. -> reject
        return False
    if "**" in equation or len(equation) > 200:  # '**' passes the char filter; 9**9**9 would hang eval
        return False
    used = [int(n) for n in re.findall(r"\d+", equation)]
    if sorted(used) != sorted(int(n) for n in nums):  # each number exactly once
        return False
    try:
        # Characters are restricted to digits/operators/parens/dot/space above,
        # so eval cannot reference any name.  Division by zero -> exception -> False.
        result = eval(equation, {"__builtins__": None}, {})
    except Exception:
        return False
    try:
        return abs(float(result) - float(target)) < 1e-6
    except Exception:
        return False


def format_reward(completions, **kwargs) -> list[float]:
    out = []
    for c in completions:
        text = _completion_text(c)
        m = _FORMAT_RE.match(text)
        if m is None:
            out.append(0.0)
        elif m.group("tail").strip() == "":
            out.append(1.0)
        else:
            out.append(0.5)
    return out


def answer_reward(completions, target, nums, **kwargs) -> list[float]:
    out = []
    for c, t, n in zip(completions, target, nums):
        eq = extract_equation(_completion_text(c))
        out.append(1.0 if equation_is_correct(eq, n, t) else 0.0)
    return out


# ----------------------------------------------------------------------------
# Self-test:  python rewards.py
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    nums, target = [44, 19, 35], 98

    good = " 44+19 = 63, 63+35 = 98. </think>\n<answer> 44 + 19 + 35 </answer>"
    junk_after = good + "\nUser: another question"
    wrong_value = " ... </think>\n<answer> 44 + 19 - 35 </answer>"
    reuse_number = " ... </think>\n<answer> 44 + 19 + 35 + 0 </answer>"        # 0 not given
    twice = " ... </think>\n<answer> 44 + 44 + 10 </answer>"                 # 44 twice
    missing_num = " ... </think>\n<answer> 63 + 35 </answer>"                 # 63 not given
    no_think_close = " 44+19+35 <answer> 44 + 19 + 35 </answer>"             # no </think>
    code_inject = " ... </think>\n<answer> __import__('os') </answer>"
    power = " ... </think>\n<answer> 44 ** 19 </answer>"                       # '**' passes regex? -> chars ok, eval huge
    division = " ... </think>\n<answer> (44 * 35) / 19 </answer>"             # 81.05 != 98
    zero_div = " ... </think>\n<answer> 44 / (19 - 19) </answer>"             # 19 twice anyway

    cases = {
        "good": (good, 1.0, 1.0),
        "junk_after": (junk_after, 0.5, 1.0),
        "wrong_value": (wrong_value, 1.0, 0.0),
        "reuse_number": (reuse_number, 1.0, 0.0),
        "twice": (twice, 1.0, 0.0),
        "missing_num": (missing_num, 1.0, 0.0),
        "no_think_close": (no_think_close, 0.0, 1.0),
        "code_inject": (code_inject, 1.0, 0.0),
        "power": (power, 1.0, 0.0),
        "division": (division, 1.0, 0.0),
        "zero_div": (zero_div, 1.0, 0.0),
    }
    comps = [v[0] for v in cases.values()]
    f = format_reward(comps)
    a = answer_reward(comps, target=[target] * len(comps), nums=[nums] * len(comps))
    ok = True
    for (name, (_, ef, ea)), gf, ga in zip(cases.items(), f, a):
        flag = "OK " if (gf == ef and ga == ea) else "FAIL"
        ok &= flag == "OK "
        print(f"{flag} {name:16s} format={gf} (want {ef})  answer={ga} (want {ea})")
    # 4-number case with division that must be exact
    print("4-num exact division:",
          answer_reward(["</think><answer>(100 / 25) * (3 + 6)</answer>"], target=[36], nums=[[100, 25, 3, 6]]))
    print("\nALL PASS" if ok else "\nSOME FAILED")


In [ ]:
%%writefile /content/grpo_puzzles/countdown/train_countdown_grpo.py
#!/usr/bin/env python
"""GRPO on the Countdown game with a *base* (not instruct) Qwen3.5 model.  No SFT.

    python train_countdown_grpo.py                       # A100-40GB defaults
    python train_countdown_grpo.py --curriculum 3only    # if the base model solves ~0% at step 0
    python train_countdown_grpo.py --lora --per_device_train_batch_size 4 --grad_accum 8   # L4 / 24 GB
    python train_countdown_grpo.py --cpu_smoke           # 2 tiny steps on CPU, wiring check only

Verified against: trl==1.12.0, transformers==5.16.1, datasets==5.0.1 (Sep 2026).

Key facts baked in (see GUIDE.md for the why):
  * We instantiate the model ourselves with AutoModelForCausalLM -> for a `qwen3_5` config this
    resolves to Qwen3_5ForCausalLM (text-only; the vision tower is simply not loaded).  If you pass
    the model *name* to GRPOTrainer instead, TRL follows config.architectures[0] and loads the full
    vision-language class + AutoProcessor, and (per TRL docs) defaults the dtype to float32.
  * Prompt = TinyZero base template ending in "<think>", plain string (TRL "standard" format).
  * Rewards: format_reward (0/0.5/1) + answer_reward (0/1). Solve rate = rewards/answer_reward/mean.
  * beta=0.0 (TRL default): no reference model, saves memory; loss_type defaults to "dapo".
  * Effective batch = per_device_train_batch_size * grad_accum completions
                    = 8 * 4 = 32 completions = 4 prompts x 8 generations per optimizer step.
"""

from __future__ import annotations

import argparse
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from data import load_countdown
from rewards import answer_reward, format_reward


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", default="Qwen/Qwen3.5-0.8B-Base")
    p.add_argument("--output_dir", default="runs/countdown-qwen3.5-0.8b-base")
    p.add_argument("--run_name", default=None)
    p.add_argument("--n_train", type=int, default=20_000, help="prompts to draw from (490k available)")
    p.add_argument("--curriculum", default="none", choices=["none", "3only", "4only"])
    p.add_argument("--max_steps", type=int, default=400)
    p.add_argument("--lr", type=float, default=1e-6, help="full FT: 5e-7..3e-6; LoRA: 1e-5..5e-5")
    p.add_argument("--beta", type=float, default=0.0, help="KL coef; 0 = no reference model")
    p.add_argument("--num_generations", type=int, default=8)
    p.add_argument("--per_device_train_batch_size", type=int, default=8)
    p.add_argument("--grad_accum", type=int, default=4)
    p.add_argument("--max_completion_length", type=int, default=512)
    p.add_argument("--temperature", type=float, default=1.0)
    p.add_argument("--lora", action="store_true", help="LoRA r=32 on all linear layers instead of full FT")
    p.add_argument("--use_vllm", action="store_true", help="colocated vLLM generation (needs vllm>=0.27, see GUIDE)")
    p.add_argument("--vllm_gpu_memory_utilization", type=float, default=0.25)
    p.add_argument("--report_to", default="none", help="'wandb' or 'none'")
    p.add_argument("--save_steps", type=int, default=100)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume", default=None, help="path to a checkpoint dir to resume from")
    p.add_argument("--cpu_smoke", action="store_true", help="tiny model, 2 steps, CPU: checks the wiring only")
    return p.parse_args()


def main():
    a = parse_args()
    on_cpu = a.cpu_smoke or not torch.cuda.is_available()

    if a.cpu_smoke:  # wiring check: tiny real model, tiny batch, 2 optimizer steps
        a.model = "HuggingFaceTB/SmolLM2-135M"
        a.max_steps, a.num_generations, a.per_device_train_batch_size, a.grad_accum = 2, 2, 2, 1
        a.max_completion_length, a.n_train, a.save_steps = 24, 8, 1000
        a.output_dir = os.path.join(a.output_dir, "cpu_smoke")

    train_ds, _eval_ds = load_countdown(n_train=a.n_train, curriculum=a.curriculum, seed=a.seed)
    print(f"train prompts: {len(train_ds)}  | example:\n{train_ds[0]['prompt']}\n")

    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"                     # GRPO pads prompts on the left
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    dtype = torch.float32 if on_cpu else torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(a.model, dtype=dtype)
    print(f"loaded {type(model).__name__} with {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

    peft_config = None
    if a.lora:
        from peft import LoraConfig
        peft_config = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.0,
                                 target_modules="all-linear", task_type="CAUSAL_LM")

    cfg = GRPOConfig(
        output_dir=a.output_dir,
        run_name=a.run_name,
        seed=a.seed,
        # --- optimisation ---
        learning_rate=a.lr,
        lr_scheduler_type="constant_with_warmup",
        warmup_steps=10,
        max_steps=a.max_steps,
        per_device_train_batch_size=a.per_device_train_batch_size,
        gradient_accumulation_steps=a.grad_accum,
        max_grad_norm=1.0,
        bf16=not on_cpu,
        use_cpu=on_cpu,
        gradient_checkpointing=not on_cpu,
        # --- GRPO ---
        num_generations=a.num_generations,
        max_completion_length=a.max_completion_length,
        temperature=a.temperature,
        beta=a.beta,
        loss_type="dapo",
        scale_rewards="group",
        # --- generation backend ---
        use_vllm=a.use_vllm,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=a.vllm_gpu_memory_utilization,
        # --- logging / saving ---
        logging_steps=1,
        log_completions=True,
        num_completions_to_print=2,
        save_steps=a.save_steps,
        save_total_limit=3,
        report_to=a.report_to,
    )

    trainer = GRPOTrainer(
        model=model,
        processing_class=tok,
        reward_funcs=[format_reward, answer_reward],
        args=cfg,
        train_dataset=train_ds,
        peft_config=peft_config,
    )
    trainer.train(resume_from_checkpoint=a.resume)

    final_dir = os.path.join(a.output_dir, "final")
    trainer.save_model(final_dir)
    tok.save_pretrained(final_dir)
    print(f"saved to {final_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grpo_puzzles/countdown/eval_countdown.py
#!/usr/bin/env python
"""Solve-rate evaluation on 500 held-out Countdown puzzles (same split every run).

    python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base            --tag base
    python eval_countdown.py --model runs/countdown-qwen3.5-0.8b-base/final --tag grpo400
    python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base --adapter runs/.../final --tag lora   # LoRA run

Greedy decoding by default (deterministic solve rate).  Add --sample --k 8 for pass@k.
Writes  results/<tag>.json  (metrics + 20 sample completions) so before/after can be plotted.
"""

from __future__ import annotations

import argparse
import json
import os
import statistics
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from data import load_countdown
from rewards import answer_reward, extract_equation, format_reward


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="Qwen/Qwen3.5-0.8B-Base")
    p.add_argument("--adapter", default=None, help="LoRA adapter dir to merge on top of --model")
    p.add_argument("--tag", required=True, help="name for results/<tag>.json")
    p.add_argument("--n", type=int, default=500, help="number of held-out puzzles (max 500)")
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--max_new_tokens", type=int, default=512)
    p.add_argument("--sample", action="store_true", help="temperature 1.0 sampling instead of greedy")
    p.add_argument("--k", type=int, default=1, help="samples per puzzle when --sample (pass@k)")
    p.add_argument("--out_dir", default="results")
    return p.parse_args()


@torch.no_grad()
def generate(model, tok, prompts, max_new_tokens, sample, batch_size):
    outs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            temperature=1.0 if sample else None,
            top_p=1.0 if sample else None,
            pad_token_id=tok.pad_token_id,
        )
        new_tokens = gen[:, enc["input_ids"].shape[1] :]
        outs.extend(tok.batch_decode(new_tokens, skip_special_tokens=True))
        print(f"  generated {min(i + batch_size, len(prompts))}/{len(prompts)}", flush=True)
    return outs


def main():
    a = parse_args()
    _, eval_ds = load_countdown(n_train=1)
    eval_ds = eval_ds.select(range(min(a.n, len(eval_ds))))

    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(a.model, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None)
    if a.adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a.adapter).merge_and_unload()
    model.eval()

    prompts = eval_ds["prompt"]
    targets, nums = eval_ds["target"], eval_ds["nums"]
    k = a.k if a.sample else 1

    t0 = time.time()
    solved_any = [0] * len(prompts)
    per_sample_solve, per_sample_format, lengths, samples = [], [], [], []
    for rep in range(k):
        comps = generate(model, tok, prompts, a.max_new_tokens, a.sample, a.batch_size)
        ar = answer_reward(comps, target=targets, nums=nums)
        fr = format_reward(comps)
        per_sample_solve.extend(ar)
        per_sample_format.extend(fr)
        lengths.extend(len(tok(c)["input_ids"]) for c in comps)
        for j, r in enumerate(ar):
            solved_any[j] = max(solved_any[j], int(r))
        if rep == 0:
            samples = [
                {"nums": n, "target": t, "completion": c, "equation": extract_equation(c), "solved": bool(r)}
                for n, t, c, r in list(zip(nums, targets, comps, ar))[:20]
            ]
    elapsed = time.time() - t0

    n3 = [i for i, n in enumerate(nums) if len(n) == 3]
    n4 = [i for i, n in enumerate(nums) if len(n) == 4]
    first = per_sample_solve[: len(prompts)]  # first sample per prompt
    metrics = {
        "tag": a.tag,
        "model": a.model,
        "adapter": a.adapter,
        "n_puzzles": len(prompts),
        "decoding": f"sample k={k}" if a.sample else "greedy",
        "solve_rate": statistics.mean(first),
        "solve_rate_3nums": statistics.mean(first[i] for i in n3) if n3 else None,
        "solve_rate_4nums": statistics.mean(first[i] for i in n4) if n4 else None,
        f"pass@{k}": statistics.mean(solved_any),
        "format_rate_clean": statistics.mean(1.0 if f == 1.0 else 0.0 for f in per_sample_format),
        "format_rate_any": statistics.mean(1.0 if f > 0 else 0.0 for f in per_sample_format),
        "mean_completion_tokens": statistics.mean(lengths),
        "seconds": round(elapsed, 1),
    }
    os.makedirs(a.out_dir, exist_ok=True)
    with open(os.path.join(a.out_dir, f"{a.tag}.json"), "w") as f:
        json.dump({"metrics": metrics, "samples": samples}, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
!python rewards.py | tail -3
!python data.py | head -5

## Optional: tokens via Colab Secrets (🔑 icon in the left sidebar)
Add `HF_TOKEN` and/or `WANDB_API_KEY` there and grant this notebook access. Nothing below *requires* them:
the baseline eval runs without any login; W&B just gives you the curves as a shareable link.

In [ ]:
import os
HF_OK = WANDB_OK = False
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"]); HF_OK = True
if os.environ.get("WANDB_API_KEY"):
    import wandb; wandb.login(key=os.environ["WANDB_API_KEY"]); WANDB_OK = True
REPORT_TO = "wandb" if WANDB_OK else "none"
print(f"HF login: {HF_OK} | W&B: {WANDB_OK} -> report_to={REPORT_TO}")

## Step 3 — baseline (go / no-go). ~10 min on an A100.
Greedy solve rate on 500 held-out puzzles, then pass@8 on 200 of them. The decision table is GUIDE.md §5.

In [ ]:
!python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base --tag base-greedy
!python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base --tag base-pass8 --sample --k 8 --n 200

In [ ]:
import json
g = json.load(open("results/base-greedy.json"))["metrics"]
p = json.load(open("results/base-pass8.json"))["metrics"]
print(f"greedy : solve={g['solve_rate']:.3f}  3nums={g['solve_rate_3nums']:.3f}  4nums={g['solve_rate_4nums']:.3f}  clean_format={g['format_rate_clean']:.3f}  any_format={g['format_rate_any']:.3f}  len={g['mean_completion_tokens']:.0f}")
print(f"pass@8 : {p['pass@8']:.3f}   (first-sample solve={p['solve_rate']:.3f}, 3nums={p['solve_rate_3nums']:.3f})")
pk = p["pass@8"]
if pk >= 0.05:   print("\n=> GO: train on the mix        (CURRICULUM = 'none')")
elif pk >= 0.01: print("\n=> THIN: 3-number puzzles first (CURRICULUM = '3only' for ~150 steps, then 'none')")
else:            print("\n=> NO SIGNAL at 0.8B-Base: try Qwen/Qwen3.5-2B-Base or Qwen/Qwen3.5-0.8B (instruct) — GUIDE.md §5")
print("\n--- 3 base-model completions ---")
for s in json.load(open("results/base-greedy.json"))["samples"][:3]:
    print(s["nums"], "->", s["target"], "| eq:", s["equation"], "| solved:", s["solved"]); print(s["completion"][:400]); print("-"*80)

## Step 4a — 5-step timing run (~5 min). Tells you the real seconds/step before you commit to 400.
Also check `frac_reward_zero_std` in the log: it must be clearly below 1.0 or GRPO has no gradient.

In [ ]:
import time; t0 = time.time()
CURRICULUM = "3only"     # from the decision above: "none" | "3only"   (2026-09-08 baseline: pass@8 = 2.5% -> 3only)
MAXLEN = 1024            # 512 clipped 98% of base completions, 1024 still clips ~90%; terminated ones average ~680 tokens
# batch 8 x 1024 tokens OOMs on the A100-40GB in the fla backward -> 4 x 8 (same 32 completions/step)
!python train_countdown_grpo.py --output_dir runs/timing --max_steps 5 --save_steps 1000 --report_to none --curriculum $CURRICULUM --max_completion_length $MAXLEN --per_device_train_batch_size 4 --grad_accum 8
print(f"{(time.time()-t0)/5:.0f} s/step incl. model load; 400 steps <= {(time.time()-t0)/5*400/3600:.1f} h")

## Step 4b — train (a few hours). First 5 steps in the log: `frac_reward_zero_std`, `rewards/answer_reward/mean`.
Batch 4 × accum 8 is already the A100 setting at 1024 tokens; on an L4 use `--lora --lr 2e-5` or `--max_completion_length 512`.

In [ ]:
import os; os.environ["WANDB_PROJECT"] = "grpo-countdown"
RUN_DIR = "runs/countdown-qwen3.5-0.8b-base"
!python train_countdown_grpo.py --output_dir $RUN_DIR --max_steps 400 --report_to $REPORT_TO --run_name cd-0.8b-base-3only-len1024 --curriculum $CURRICULUM --max_completion_length $MAXLEN --per_device_train_batch_size 4 --grad_accum 8 --save_steps 50

## Step 5 — curves (from the checkpoint's `trainer_state.json`; W&B has the full set)

In [ ]:
KEYS = ["rewards/answer_reward/mean", "rewards/format_reward/mean", "completions/mean_length", "frac_reward_zero_std"]
TITLES = ["solve rate (train prompts)", "format reward", "completion length", "dead groups (all-same reward)"]
import json, glob, os
import matplotlib.pyplot as plt

def latest_state(run_dir):
    ck = sorted(glob.glob(f"{run_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    path = os.path.join(ck[-1], "trainer_state.json") if ck else os.path.join(run_dir, "final", "trainer_state.json")
    return json.load(open(path))["log_history"]

hist = latest_state(RUN_DIR)
def series(key):
    return [(h["step"], h[key]) for h in hist if key in h]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, key, title in zip(axes, KEYS, TITLES):
    s = series(key)
    if s:
        ax.plot(*zip(*s)); ax.set_title(title); ax.set_xlabel("optimizer step"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RUN_DIR}/curves.png", dpi=150); plt.show()
print("saved", f"{RUN_DIR}/curves.png")

## Step 6 — after: same 500 held-out puzzles, before vs after

In [ ]:
!python eval_countdown.py --model $RUN_DIR/final --tag grpo400
!python eval_countdown.py --model $RUN_DIR/final --tag grpo400-pass8 --sample --k 8 --n 200
import json
for tag in ["base-greedy", "grpo400"]:
    m = json.load(open(f"results/{tag}.json"))["metrics"]
    print(f"{tag:12s} solve={m['solve_rate']:.3f}  3nums={m['solve_rate_3nums']:.3f}  4nums={m['solve_rate_4nums']:.3f}  clean_format={m['format_rate_clean']:.3f}  len={m['mean_completion_tokens']:.0f}")

In [ ]:
# look at what it learned to say
import json
for s in json.load(open("results/grpo400.json"))["samples"][:5]:
    print(s["nums"], "->", s["target"], "| solved:", s["solved"]); print(s["completion"][:600]); print("-"*80)

## Keep the outputs (Colab disks vanish on disconnect)
Either mount Drive and copy `results/` + `curves.png` + `final/`, or download them from the file browser.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/grpo_puzzles && cp -r results $RUN_DIR/curves.png /content/drive/MyDrive/grpo_puzzles/
# Publish the model (needs HF_TOKEN secret):
# from transformers import AutoModelForCausalLM, AutoTokenizer
# AutoModelForCausalLM.from_pretrained(f"{RUN_DIR}/final").push_to_hub("<you>/qwen3.5-0.8b-countdown-grpo")
# AutoTokenizer.from_pretrained(f"{RUN_DIR}/final").push_to_hub("<you>/qwen3.5-0.8b-countdown-grpo")